# 🏛️ TinyLlama Oracle LoRA — NLP → SQL Donnée Audit Oracle

**Workflow complet production : entraînement + tests interactifs avec scoring**

| Étape | Cellule | Description |
|---|---|---|
| Installation | 1 | pip install + redémarrage auto |
| Vérification | 2 | Versions + GPU |
| Modèle | 3 | Téléchargement TinyLlama |
| Table AUDIT | 4 | Oracle Audit Trail enrichi (500 lignes) |
| Dataset | 5 | 3000 exemples FR↔SQL |
| Entraînement | 6 | Fine-tuning LoRA |
| Chargement | 7 | Inférence LoRA |
| Aperçu | 8 | Table AUDIT + guide questions |
| Test FR→SQL | 9 | Tests interactifs avec scoring |
| Test SQL→FR | 10 | Reformulation résultats Oracle |
| Bilan | 11 | Rapport global de performance |
| Production | 12 | Guide déploiement |

> ⚠️ **Exécuter cellule 1 seule** puis attendre le redémarrage automatique.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 1 — Installation des dépendances                   ║
# ║  Exécuter SEULE puis attendre le redémarrage automatique     ║
# ╚══════════════════════════════════════════════════════════════╝

# Étape 1 : purger numpy corrompu
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","uninstall","-y","numpy"], capture_output=True)
subprocess.run([sys.executable,"-m","pip","cache","purge"], capture_output=True)

# Étape 2 : numpy dans la plage compatible torch + tensorflow + numba
subprocess.run([sys.executable,"-m","pip","install","-q","--no-cache-dir",
               "numpy>=2.0.0,<2.1.0"], check=True)

# Étape 3 : stack ML
subprocess.run([sys.executable,"-m","pip","install","-q","--no-cache-dir",
               "transformers>=4.40.0","peft>=0.10.0","accelerate>=0.28.0",
               "datasets>=2.19.0","torch>=2.2.0","huggingface_hub>=0.22.0",
               "pandas>=2.0.0","tabulate"], check=True)

print("\n✅ Installation terminée — redémarrage automatique...")
import os; os.kill(os.getpid(), 9)

In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 2 — Vérification des versions                      ║
# ╚══════════════════════════════════════════════════════════════╝
import sys, numpy as np, torch, transformers, peft, accelerate, datasets

print("="*55)
print("  ENVIRONNEMENT COLAB")
print("="*55)
print(f"  Python       : {sys.version.split()[0]}")
print(f"  NumPy        : {np.__version__}")
print(f"  PyTorch      : {torch.__version__}")
print(f"  Transformers : {transformers.__version__}")
print(f"  PEFT         : {peft.__version__}")
print(f"  Accelerate   : {accelerate.__version__}")
print(f"  Datasets     : {datasets.__version__}")
print("="*55)
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    print(f"  GPU          : {dev.name}")
    print(f"  VRAM         : {dev.total_memory/1e9:.1f} GB")
    print("  ✅ GPU prêt pour l'entraînement")
else:
    print("  ⚠️  Aucun GPU — activer dans Exécution › Modifier le type d'exécution")
print("="*55)

  ENVIRONNEMENT COLAB
  Python       : 3.12.12
  NumPy        : 2.0.2
  PyTorch      : 2.10.0+cu128
  Transformers : 5.0.0
  PEFT         : 0.18.1
  Accelerate   : 1.12.0
  Datasets     : 4.0.0
  GPU          : Tesla T4
  VRAM         : 15.6 GB
  ✅ GPU prêt pour l'entraînement


In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 3 — Téléchargement TinyLlama                       ║
# ╚══════════════════════════════════════════════════════════════╝
from huggingface_hub import snapshot_download
import shutil, os

MODEL_DIR = "TinyLlama-1.1B-Chat-v1.0"

if not os.path.exists(MODEL_DIR):
    print("📥 Téléchargement TinyLlama-1.1B-Chat-v1.0 (~2.2 GB)...")
    snapshot_download(repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                      local_dir=MODEL_DIR, local_dir_use_symlinks=False)
    print("✅ Modèle téléchargé.")
else:
    print(f"✅ Modèle déjà présent : {MODEL_DIR}")

if not os.path.exists(MODEL_DIR + ".zip"):
    shutil.make_archive(MODEL_DIR, "zip", MODEL_DIR)
    print(f"📦 Archive : {MODEL_DIR}.zip")

📥 Téléchargement TinyLlama-1.1B-Chat-v1.0 (~2.2 GB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

✅ Modèle téléchargé.
📦 Archive : TinyLlama-1.1B-Chat-v1.0.zip


In [3]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 4 — Table ORACLE_AUDIT_TRAIL (500 lignes)           ║
# ║  Distribution réaliste : ~70% SELECT, ~20% connexions        ║
# ║  Structure basée sur DBA_AUDIT_TRAIL / UNIFIED_AUDIT_TRAIL   ║
# ╚══════════════════════════════════════════════════════════════╝
import random, pandas as pd
from datetime import datetime, timedelta

random.seed(42)

OS_USERS   = ["oracle","sys","system","dbadmin","appuser","reports_user",
              "audit_user","dba_ops","backup_agent","monitor_svc"]
DB_USERS   = ["SYS","SYSTEM","SCOTT","HR","APP_USER","REPORT_USR",
              "AUDIT_USR","DBA_OPS","BACKUP_USR","MONITOR"]
SCHEMAS    = ["SYS","HR","SCOTT","APP_SCHEMA","AUDIT_SCHEMA"]
OBJECTS    = ["EMP","DEPT","SALGRADE","FACTURE","CLIENT","TRANSACTION",
              "AUDIT_LOG","V$SESSION","V$SQL","DBA_USERS","DBA_OBJECTS",
              "ALL_TABLES","PAYROLL","ORDERS","CUSTOMERS","PRODUCTS",
              "ACCOUNTS","JOURNAL","BUDGET","CONTRACTS"]
TERMINALS  = ["pts/0","pts/1","pts/2","pts/3","console","UNKNOWN","tty1"]
HOSTS      = ["srv-oracle-01","srv-oracle-02","app-server-01",
              "app-server-02","backup-srv","monitor-01","report-srv"]
PRIVILEGES = ["SELECT ANY TABLE","CREATE SESSION","DBA","SYSDBA",
              "CREATE TABLE","EXECUTE ANY PROCEDURE","","","","","",""]
RETURNCODES= [0,0,0,0,0,0,0,0,0,0,0,0,1,1017,904,942,955,1031]

SQL_TEXTS_SELECT = [
    "SELECT * FROM EMP WHERE DEPTNO=10",
    "SELECT COUNT(*) FROM FACTURE WHERE STATUT='NON_PAYE'",
    "SELECT E.ENAME, D.DNAME FROM EMP E JOIN DEPT D ON E.DEPTNO=D.DEPTNO",
    "SELECT SUM(SAL) FROM EMP GROUP BY DEPTNO",
    "SELECT * FROM V$SESSION WHERE STATUS='ACTIVE'",
    "SELECT * FROM DBA_USERS WHERE ACCOUNT_STATUS='OPEN'",
    "SELECT * FROM ALL_TABLES WHERE OWNER='HR'",
    "SELECT * FROM PAYROLL WHERE SALARY > 5000",
    "SELECT USER, COUNT(*) FROM AUDIT_LOG GROUP BY USER",
    "SELECT * FROM CONTRACTS WHERE EXPIRY_DATE < SYSDATE",
    "SELECT * FROM BUDGET WHERE YEAR=2026",
    "SELECT SYSDATE FROM DUAL",
    "SELECT * FROM ORDERS WHERE ROWNUM <= 100",
    "SELECT AVG(SAL), DEPTNO FROM EMP GROUP BY DEPTNO",
    "SELECT COUNT(*) FROM CUSTOMERS WHERE CITY='Paris'",
    "SELECT MAX(MONTANT) FROM TRANSACTION",
    "SELECT * FROM ACCOUNTS WHERE BALANCE < 0",
    "SELECT ENAME, SAL FROM EMP ORDER BY SAL DESC",
    "SELECT DISTINCT STATUS FROM V$SESSION",
    "SELECT * FROM DBA_OBJECTS WHERE OBJECT_TYPE='TABLE'",
]
SQL_TEXTS_OTHER = [
    "UPDATE FACTURE SET STATUT='PAYE' WHERE ID=2045",
    "DELETE FROM AUDIT_LOG WHERE TIMESTAMP < SYSDATE-90",
    "INSERT INTO ORDERS VALUES(:1,:2,:3,:4)",
    "CREATE TABLE TEMP_REPORT AS SELECT * FROM ORDERS WHERE ROWNUM<1000",
    "DROP TABLE TEMP_REPORT",
    "GRANT SELECT ON EMP TO REPORT_USR",
    "REVOKE DBA FROM SCOTT",
    "EXECUTE DBMS_STATS.GATHER_TABLE_STATS('HR','EMP')",
    "ALTER TABLE FACTURE ADD CONSTRAINT FK_CLI FOREIGN KEY(CLIENT_ID) REFERENCES CLIENT(ID)",
    "TRUNCATE TABLE TEMP_DATA",
    "CREATE INDEX IDX_FACTURE_STATUT ON FACTURE(STATUT)",
]
COMMENTS = [
    "Routine reporting query","Scheduled backup procedure",
    "Application login via connection pool","Automated monitoring script",
    "DBA maintenance task","End of day reconciliation",
    "","Emergency access - incident #4521","Performance audit",
    "Compliance check Q1-2026","","",
]

# ── Distribution réaliste ─────────────────────────────────
# 70% SELECT, 15% LOGON/LOGOFF, 10% agrégations/rapports, 5% admin
ACTION_POOL = (
    ["SELECT"] * 70 +
    ["LOGON"] * 10 +
    ["LOGOFF"] * 8 +
    ["INSERT"] * 3 +
    ["UPDATE"] * 3 +
    ["DELETE"] * 2 +
    ["EXECUTE"] * 2 +
    ["GRANT"] * 1 +
    ["REVOKE"] * 1
)

start = datetime(2026, 1, 1, 6, 0, 0)
rows = []
for i in range(1, 501):
    ts = start + timedelta(
        days=random.randint(0, 55),
        hours=random.randint(0, 17),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59)
    )
    action = random.choice(ACTION_POOL)
    rcode  = random.choice(RETURNCODES)
    is_select = action == "SELECT"
    is_conn   = action in ["LOGON","LOGOFF"]

    obj    = "" if is_conn else random.choice(OBJECTS)
    schema = "" if is_conn else random.choice(SCHEMAS)
    sql    = (random.choice(SQL_TEXTS_SELECT) if is_select
              else ("" if is_conn else random.choice(SQL_TEXTS_OTHER)))

    rows.append({
        "AUDIT_ID"      : i,
        "TIMESTAMP"     : ts.strftime("%Y-%m-%d %H:%M:%S"),
        "OS_USER"       : random.choice(OS_USERS),
        "DB_USER"       : random.choice(DB_USERS),
        "OS_HOST"       : random.choice(HOSTS),
        "TERMINAL"      : random.choice(TERMINALS),
        "SESSIONID"     : random.randint(100, 9999),
        "ACTION_NAME"   : action,
        "OBJ_SCHEMA"    : schema,
        "OBJ_NAME"      : obj,
        "SQL_TEXT"      : sql,
        "RETURNCODE"    : rcode,
        "PRIVILEGE_USED": random.choice(PRIVILEGES),
        "STATEMENT"     : i * 3 + random.randint(1,5),
        "COMMENT_TEXT"  : random.choice(COMMENTS),
    })

AUDIT_DF = pd.DataFrame(rows)
AUDIT_DF.to_csv("oracle_audit_trail.csv", index=False)

print("="*65)
print("  TABLE : ORACLE_AUDIT_TRAIL  (basée sur DBA_AUDIT_TRAIL)")
print("="*65)
print(f"  Lignes   : {len(AUDIT_DF)}")
print(f"  Colonnes : {len(AUDIT_DF.columns)}")
print(f"  Période  : {AUDIT_DF.TIMESTAMP.min()} → {AUDIT_DF.TIMESTAMP.max()}")
print()
print("  COLONNES DISPONIBLES :")
col_descriptions = {
    "AUDIT_ID"      : "Identifiant unique de l'entrée d'audit",
    "TIMESTAMP"     : "Date et heure de l'événement",
    "OS_USER"       : "Utilisateur système Linux",
    "DB_USER"       : "Utilisateur Oracle",
    "OS_HOST"       : "Nom du serveur source",
    "TERMINAL"      : "Terminal utilisé (pts/0, console…)",
    "SESSIONID"     : "Identifiant de session Oracle",
    "ACTION_NAME"   : "Type d'action (SELECT, LOGON, UPDATE…)",
    "OBJ_SCHEMA"    : "Schéma de l'objet cible",
    "OBJ_NAME"      : "Nom de l'objet cible (table, vue…)",
    "SQL_TEXT"      : "Texte SQL exécuté",
    "RETURNCODE"    : "Code retour Oracle (0=succès, >0=erreur ORA-)",
    "PRIVILEGE_USED": "Privilège Oracle utilisé",
    "STATEMENT"     : "Numéro de statement Oracle",
    "COMMENT_TEXT"  : "Commentaire d'audit",
}
for col in AUDIT_DF.columns:
    print(f"    {col:<20} : {col_descriptions.get(col,'')}")

print()
print("  DISTRIBUTION DES ACTIONS (réaliste) :")
dist = AUDIT_DF.ACTION_NAME.value_counts()
total = len(AUDIT_DF)
for action, count in dist.items():
    pct = count/total*100
    bar = "█" * (count // 4)
    print(f"    {action:<20} {count:>4}  ({pct:>4.1f}%)  {bar}")

print()
print("  CODES RETOUR :")
rc = AUDIT_DF.RETURNCODE.value_counts().head(8)
for code, count in rc.items():
    label = "✅ Succès" if code == 0 else f"❌ ORA-{code:05d}"
    print(f"    {label:<25} {count:>4} occurrences")

print()
print("  APERÇU (5 lignes) :")
print(AUDIT_DF[["AUDIT_ID","TIMESTAMP","DB_USER","ACTION_NAME",
                "OBJ_NAME","RETURNCODE"]].head(5).to_string(index=False))
print()
print("✅ Table AUDIT enrichie — distribution réaliste (70% SELECT)")
print("   Fichier sauvegardé : oracle_audit_trail.csv")

  TABLE : ORACLE_AUDIT_TRAIL  (basée sur DBA_AUDIT_TRAIL)
  Lignes   : 500
  Colonnes : 15
  Période  : 2026-01-01 07:18:38 → 2026-02-25 23:56:22

  COLONNES DISPONIBLES :
    AUDIT_ID             : Identifiant unique de l'entrée d'audit
    TIMESTAMP            : Date et heure de l'événement
    OS_USER              : Utilisateur système Linux
    DB_USER              : Utilisateur Oracle
    OS_HOST              : Nom du serveur source
    TERMINAL             : Terminal utilisé (pts/0, console…)
    SESSIONID            : Identifiant de session Oracle
    ACTION_NAME          : Type d'action (SELECT, LOGON, UPDATE…)
    OBJ_SCHEMA           : Schéma de l'objet cible
    OBJ_NAME             : Nom de l'objet cible (table, vue…)
    SQL_TEXT             : Texte SQL exécuté
    RETURNCODE           : Code retour Oracle (0=succès, >0=erreur ORA-)
    PRIVILEGE_USED       : Privilège Oracle utilisé
    STATEMENT            : Numéro de statement Oracle
    COMMENT_TEXT         : Commentai

In [4]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 5 — Dataset 4000 exemples                          ║
# ║  Distribution FR→SQL calquée sur l'usage réel :             ║
# ║    ~50% SELECT/filtres   ~25% agrégations                   ║
# ║    ~15% sous-requêtes    ~10% avancé/sécu/mixte             ║
# ╚══════════════════════════════════════════════════════════════╝
import pandas as pd, random
random.seed(42)

ACTIONS_SELECT = ["SELECT","LOGON","LOGOFF","INSERT","UPDATE","DELETE",
                  "GRANT","REVOKE","EXECUTE","CREATE TABLE","DROP TABLE"]
DB_USERS = ["SYS","SYSTEM","SCOTT","HR","APP_USER","REPORT_USR",
            "AUDIT_USR","DBA_OPS","BACKUP_USR","MONITOR"]
OS_HOSTS = ["srv-oracle-01","srv-oracle-02","app-server-01",
            "backup-srv","monitor-01","report-srv","admin-ws-01"]
OBJECTS  = ["EMP","DEPT","FACTURE","CLIENT","TRANSACTION","PAYROLL",
            "ORDERS","ACCOUNTS","AUDIT_LOG","V$SESSION","DBA_USERS",
            "ALL_TABLES","CONTRACTS","BUDGET","JOURNAL"]
DATES    = [f"2026-{m:02d}-{d:02d}"
            for m in range(1,3) for d in range(1,29) if not (m==2 and d>28)]

# ════════════════════════════════════════════════════════════
# CAT A — SELECT / Filtres simples   CIBLE : 700 exemples
# ════════════════════════════════════════════════════════════
cat_a = []

# A1 — questions naturelles sur les SELECT (action la plus commune)
select_questions = [
    ("Quels utilisateurs Oracle ont effectué des requêtes SELECT dans l'audit ?",
     "SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' ORDER BY DB_USER;"),
    ("Combien de SELECT ont été enregistrés dans l'audit Oracle ?",
     "SELECT COUNT(*) AS NB_SELECT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT';"),
    ("Quels objets Oracle ont été consultés en SELECT ?",
     "SELECT DISTINCT OBJ_NAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND OBJ_NAME!='' GROUP BY OBJ_NAME ORDER BY NB DESC;"),
    ("Quel utilisateur Oracle a fait le plus de SELECT dans l'audit ?",
     "SELECT DB_USER, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DB_USER ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Liste tous les SELECT avec la date et l'utilisateur.",
     "SELECT AUDIT_ID, TIMESTAMP, DB_USER, OBJ_NAME, SQL_TEXT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' ORDER BY TIMESTAMP;"),
    ("Quels SELECT ont été faits sur la table EMP ?",
     "SELECT AUDIT_ID, TIMESTAMP, DB_USER, SQL_TEXT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND OBJ_NAME='EMP' ORDER BY TIMESTAMP;"),
    ("Combien de SELECT par utilisateur Oracle dans l'audit ?",
     "SELECT DB_USER, COUNT(*) AS NB_SELECT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DB_USER ORDER BY NB_SELECT DESC;"),
    ("Quels SELECT ont échoué (code retour non nul) dans l'audit ?",
     "SELECT AUDIT_ID, TIMESTAMP, DB_USER, OBJ_NAME, RETURNCODE FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND RETURNCODE!=0;"),
    ("Liste les SELECT effectués depuis le serveur srv-oracle-01.",
     "SELECT AUDIT_ID, TIMESTAMP, DB_USER, OBJ_NAME FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND OS_HOST='srv-oracle-01' ORDER BY TIMESTAMP;"),
    ("Quels utilisateurs ont fait des SELECT sur des vues système (DBA_USERS, V$SESSION) ?",
     "SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND OBJ_NAME IN ('DBA_USERS','V$SESSION','ALL_TABLES','DBA_OBJECTS');"),
    ("Montre-moi tous les SELECT du mois de janvier 2026.",
     "SELECT AUDIT_ID, TIMESTAMP, DB_USER, OBJ_NAME FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND TIMESTAMP LIKE '2026-01%' ORDER BY TIMESTAMP;"),
    ("Quel est le dernier SELECT enregistré dans l'audit Oracle ?",
     "SELECT AUDIT_ID, TIMESTAMP, DB_USER, OBJ_NAME, SQL_TEXT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' ORDER BY TIMESTAMP DESC FETCH FIRST 1 ROWS ONLY;"),
]

# Multiplier par reformulations pour atteindre la cible
reformulations_a = [
    ("Quels utilisateurs","Lister les utilisateurs"),
    ("Combien de","Donner le nombre de"),
    ("Liste","Afficher"),
    ("Montre-moi","Afficher"),
    ("Quel utilisateur","Identifier l'utilisateur"),
    ("Quels objets","Lister les objets"),
    ("Quels SELECT","Afficher les SELECT"),
]
for instr, sql in select_questions:
    cat_a.append({"instruction": instr, "output": sql})
    for old, new in reformulations_a:
        if old in instr:
            cat_a.append({"instruction": instr.replace(old, new, 1), "output": sql})
            break

# A2 — par utilisateur DB (10 users × 5 questions = 50)
for user in DB_USERS:
    cat_a += [
        {"instruction": f"Quelles requêtes SELECT l'utilisateur {user} a-t-il effectuées ?",
         "output": f"SELECT AUDIT_ID, TIMESTAMP, OBJ_NAME, SQL_TEXT FROM ORACLE_AUDIT_TRAIL WHERE DB_USER='{user}' AND ACTION_NAME='SELECT' ORDER BY TIMESTAMP;"},
        {"instruction": f"Combien de SELECT l'utilisateur {user} a-t-il fait dans l'audit ?",
         "output": f"SELECT COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE DB_USER='{user}' AND ACTION_NAME='SELECT';"},
        {"instruction": f"Quels objets l'utilisateur {user} a-t-il consultés en SELECT ?",
         "output": f"SELECT DISTINCT OBJ_NAME FROM ORACLE_AUDIT_TRAIL WHERE DB_USER='{user}' AND ACTION_NAME='SELECT' AND OBJ_NAME IS NOT NULL AND OBJ_NAME!='' ORDER BY OBJ_NAME;"},
        {"instruction": f"L'utilisateur {user} a-t-il effectué des SELECT avec des erreurs Oracle ?",
         "output": f"SELECT AUDIT_ID, TIMESTAMP, OBJ_NAME, RETURNCODE FROM ORACLE_AUDIT_TRAIL WHERE DB_USER='{user}' AND ACTION_NAME='SELECT' AND RETURNCODE!=0;"},
        {"instruction": f"Derniers SELECT de l'utilisateur Oracle {user}.",
         "output": f"SELECT AUDIT_ID, TIMESTAMP, OBJ_NAME, SQL_TEXT FROM ORACLE_AUDIT_TRAIL WHERE DB_USER='{user}' AND ACTION_NAME='SELECT' ORDER BY TIMESTAMP DESC FETCH FIRST 10 ROWS ONLY;"},
    ]

# A3 — par objet (15 objets × 4 questions = 60)
for obj in OBJECTS:
    cat_a += [
        {"instruction": f"Qui a effectué des SELECT sur {obj} dans l'audit Oracle ?",
         "output": f"SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE OBJ_NAME='{obj}' AND ACTION_NAME='SELECT' ORDER BY DB_USER;"},
        {"instruction": f"Combien de SELECT sur {obj} dans l'audit ?",
         "output": f"SELECT COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE OBJ_NAME='{obj}' AND ACTION_NAME='SELECT';"},
        {"instruction": f"Quand a-t-on accédé à {obj} pour la dernière fois en SELECT ?",
         "output": f"SELECT MAX(TIMESTAMP) AS DERNIER_SELECT FROM ORACLE_AUDIT_TRAIL WHERE OBJ_NAME='{obj}' AND ACTION_NAME='SELECT';"},
        {"instruction": f"Liste tous les accès SELECT à {obj} avec date et utilisateur.",
         "output": f"SELECT AUDIT_ID, TIMESTAMP, DB_USER, SQL_TEXT FROM ORACLE_AUDIT_TRAIL WHERE OBJ_NAME='{obj}' AND ACTION_NAME='SELECT' ORDER BY TIMESTAMP;"},
    ]

# A4 — par date (15 dates × 3 = 45)
for date in random.sample(DATES, 15):
    cat_a += [
        {"instruction": f"Quels SELECT ont été effectués le {date} dans l'audit Oracle ?",
         "output": f"SELECT DB_USER, OBJ_NAME, SQL_TEXT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND TIMESTAMP LIKE '{date}%' ORDER BY TIMESTAMP;"},
        {"instruction": f"Combien de SELECT le {date} dans l'audit ?",
         "output": f"SELECT COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND TIMESTAMP LIKE '{date}%';"},
        {"instruction": f"Qui a consulté la base Oracle le {date} ?",
         "output": f"SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE TIMESTAMP LIKE '{date}%' ORDER BY DB_USER;"},
    ]

# A5 — connexions / sessions (important pour l'audit)
connexion_questions = [
    ("Quels utilisateurs Oracle se sont connectés dans l'audit ?",
     "SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' ORDER BY DB_USER;"),
    ("Combien de connexions LOGON dans l'audit Oracle ?",
     "SELECT COUNT(*) AS NB_CONNEXIONS FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON';"),
    ("Quels serveurs ont servi de point de connexion Oracle ?",
     "SELECT DISTINCT OS_HOST FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' ORDER BY OS_HOST;"),
    ("Nombre de connexions par utilisateur Oracle dans l'audit.",
     "SELECT DB_USER, COUNT(*) AS NB_LOGON FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' GROUP BY DB_USER ORDER BY NB_LOGON DESC;"),
    ("Quand SYS s'est-il connecté à Oracle dans l'audit ?",
     "SELECT TIMESTAMP, OS_HOST, TERMINAL FROM ORACLE_AUDIT_TRAIL WHERE DB_USER='SYS' AND ACTION_NAME='LOGON' ORDER BY TIMESTAMP;"),
    ("Y a-t-il des connexions LOGON sans LOGOFF correspondant ?",
     "SELECT DB_USER, SESSIONID FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' AND SESSIONID NOT IN (SELECT SESSIONID FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGOFF');"),
    ("Combien de déconnexions LOGOFF dans l'audit Oracle ?",
     "SELECT COUNT(*) AS NB_LOGOFF FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGOFF';"),
    ("Quels utilisateurs se sont connectés depuis app-server-01 ?",
     "SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' AND OS_HOST='app-server-01' ORDER BY DB_USER;"),
]
for instr, sql in connexion_questions:
    cat_a.append({"instruction": instr, "output": sql})
    for old, new in reformulations_a:
        if old in instr:
            cat_a.append({"instruction": instr.replace(old, new, 1), "output": sql})
            break

# A6 — codes retour / erreurs
erreur_questions = [
    ("Quels événements d'audit se sont soldés par une erreur Oracle ?",
     "SELECT AUDIT_ID, TIMESTAMP, DB_USER, ACTION_NAME, RETURNCODE FROM ORACLE_AUDIT_TRAIL WHERE RETURNCODE!=0 ORDER BY TIMESTAMP;"),
    ("Combien d'erreurs Oracle sont enregistrées dans l'audit ?",
     "SELECT COUNT(*) AS NB_ERREURS FROM ORACLE_AUDIT_TRAIL WHERE RETURNCODE!=0;"),
    ("Quels utilisateurs ont généré des erreurs ORA-01017 dans l'audit ?",
     "SELECT DISTINCT DB_USER, OS_HOST FROM ORACLE_AUDIT_TRAIL WHERE RETURNCODE=1017;"),
    ("Quels SELECT ont échoué avec une erreur ORA-00942 (table inexistante) ?",
     "SELECT AUDIT_ID, TIMESTAMP, DB_USER, OBJ_NAME FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND RETURNCODE=942;"),
    ("Taux de succès des requêtes SELECT dans l'audit Oracle.",
     "SELECT ROUND(SUM(CASE WHEN RETURNCODE=0 THEN 1 ELSE 0 END)*100/COUNT(*),2) AS TAUX_SUCCES FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT';"),
    ("Nombre d'erreurs par type de code retour Oracle.",
     "SELECT RETURNCODE, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE RETURNCODE!=0 GROUP BY RETURNCODE ORDER BY NB DESC;"),
]
for instr, sql in erreur_questions:
    cat_a.append({"instruction": instr, "output": sql})
    cat_a.append({"instruction": instr.replace("Quels","Identifier les").replace("Combien","Donner le nombre"), "output": sql})

# Compléter jusqu'à 700
base_a = cat_a[:]
while len(cat_a) < 700:
    ex = random.choice(base_a)
    pfxs = ["","Afficher : ","Rapport : ","Consulter : ","Analyse : "]
    ni = random.choice(pfxs) + ex["instruction"].lower()
    cat_a.append({"instruction": ni.strip().capitalize(), "output": ex["output"]})
cat_a = cat_a[:700]

# ════════════════════════════════════════════════════════════
# CAT B — Agrégations GROUP BY / HAVING   CIBLE : 500
# ════════════════════════════════════════════════════════════
cat_b = []

agg_questions = [
    ("Nombre de SELECT par utilisateur Oracle dans l'audit.",
     "SELECT DB_USER, COUNT(*) AS NB_SELECT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DB_USER ORDER BY NB_SELECT DESC;"),
    ("Quel utilisateur Oracle a fait le plus de requêtes dans l'audit ?",
     "SELECT DB_USER, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY DB_USER ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Volume total d'activité par utilisateur Oracle dans l'audit.",
     "SELECT DB_USER, COUNT(*) AS NB_TOTAL FROM ORACLE_AUDIT_TRAIL GROUP BY DB_USER ORDER BY NB_TOTAL DESC;"),
    ("Quels objets Oracle sont les plus consultés en SELECT ?",
     "SELECT OBJ_NAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND OBJ_NAME!='' GROUP BY OBJ_NAME ORDER BY NB DESC FETCH FIRST 10 ROWS ONLY;"),
    ("Activité Oracle par jour dans l'audit.",
     "SELECT SUBSTR(TIMESTAMP,1,10) AS JOUR, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY SUBSTR(TIMESTAMP,1,10) ORDER BY JOUR;"),
    ("Activité Oracle par heure de la journée.",
     "SELECT TO_NUMBER(SUBSTR(TIMESTAMP,12,2)) AS HEURE, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY TO_NUMBER(SUBSTR(TIMESTAMP,12,2)) ORDER BY HEURE;"),
    ("Nombre de SELECT par objet Oracle dans l'audit.",
     "SELECT OBJ_NAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY OBJ_NAME ORDER BY NB DESC;"),
    ("Quels serveurs Oracle génèrent le plus d'activité SELECT ?",
     "SELECT OS_HOST, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY OS_HOST ORDER BY NB DESC;"),
    ("Résumé global audit Oracle : total, succès, erreurs.",
     "SELECT COUNT(*) AS TOTAL, SUM(CASE WHEN RETURNCODE=0 THEN 1 ELSE 0 END) AS SUCCES, SUM(CASE WHEN RETURNCODE!=0 THEN 1 ELSE 0 END) AS ERREURS FROM ORACLE_AUDIT_TRAIL;"),
    ("Nombre de connexions par jour dans l'audit Oracle.",
     "SELECT SUBSTR(TIMESTAMP,1,10) AS JOUR, COUNT(*) AS NB_LOGON FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' GROUP BY SUBSTR(TIMESTAMP,1,10) ORDER BY JOUR;"),
    ("Quels utilisateurs ont effectué plus de 10 SELECT dans l'audit ?",
     "SELECT DB_USER, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DB_USER HAVING COUNT(*) > 10 ORDER BY NB DESC;"),
    ("Taux d'erreur par utilisateur Oracle dans l'audit.",
     "SELECT DB_USER, ROUND(SUM(CASE WHEN RETURNCODE!=0 THEN 1 ELSE 0 END)*100/COUNT(*),2) AS TAUX_ERR FROM ORACLE_AUDIT_TRAIL GROUP BY DB_USER ORDER BY TAUX_ERR DESC;"),
    ("Nombre de SELECT par schéma Oracle dans l'audit.",
     "SELECT OBJ_SCHEMA, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND OBJ_SCHEMA!='' GROUP BY OBJ_SCHEMA ORDER BY NB DESC;"),
    ("Distribution des actions Oracle dans l'audit.",
     "SELECT ACTION_NAME, COUNT(*) AS NB, ROUND(COUNT(*)*100/SUM(COUNT(*)) OVER(),1) AS PCT FROM ORACLE_AUDIT_TRAIL GROUP BY ACTION_NAME ORDER BY NB DESC;"),
    ("Quels utilisateurs ont accédé à plus de 5 objets distincts en SELECT ?",
     "SELECT DB_USER, COUNT(DISTINCT OBJ_NAME) AS NB_OBJETS FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DB_USER HAVING COUNT(DISTINCT OBJ_NAME) > 5 ORDER BY NB_OBJETS DESC;"),
    ("Nombre moyen de SELECT par utilisateur Oracle dans l'audit.",
     "SELECT AVG(NB) AS MOYENNE FROM (SELECT COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DB_USER);"),
    ("Heure de pointe des SELECT Oracle dans l'audit.",
     "SELECT TO_NUMBER(SUBSTR(TIMESTAMP,12,2)) AS HEURE, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY TO_NUMBER(SUBSTR(TIMESTAMP,12,2)) ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("Percentage de SELECT vs autres actions dans l'audit Oracle.",
     "SELECT ACTION_NAME, COUNT(*) AS NB, ROUND(COUNT(*)*100/SUM(COUNT(*)) OVER(),2) AS PCT FROM ORACLE_AUDIT_TRAIL GROUP BY ACTION_NAME ORDER BY NB DESC;"),
    ("Nombre d'objets distincts consultés par utilisateur Oracle.",
     "SELECT DB_USER, COUNT(DISTINCT OBJ_NAME) AS NB_OBJ FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DB_USER ORDER BY NB_OBJ DESC;"),
    ("Activité SELECT Oracle par semaine dans l'audit.",
     "SELECT TO_CHAR(TO_DATE(SUBSTR(TIMESTAMP,1,10),'YYYY-MM-DD'),'IYYY-IW') AS SEMAINE, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY TO_CHAR(TO_DATE(SUBSTR(TIMESTAMP,1,10),'YYYY-MM-DD'),'IYYY-IW') ORDER BY SEMAINE;"),
]

reformulations_b = [
    ("Nombre","Compter le nombre"),("Quel utilisateur","Identifier l'utilisateur"),
    ("Quels","Lister les"),("Volume","Calculer le volume"),
    ("Distribution","Répartition"),("Résumé","Bilan"),
]
for instr, sql in agg_questions:
    cat_b.append({"instruction": instr, "output": sql})
    for old, new in reformulations_b:
        if old in instr:
            cat_b.append({"instruction": instr.replace(old, new, 1), "output": sql})
            cat_b.append({"instruction": "Statistique Oracle : " + instr.lower(), "output": sql})
            break

base_b = cat_b[:]
while len(cat_b) < 500:
    ex = random.choice(base_b)
    pfx = random.choice(["Rapport : ","Analyse : ","KPI Oracle : ","Métriques : "])
    ni = pfx + ex["instruction"].lower()
    cat_b.append({"instruction": ni.capitalize(), "output": ex["output"]})
cat_b = cat_b[:500]

# ════════════════════════════════════════════════════════════
# CAT C — Sous-requêtes   CIBLE : 300
# ════════════════════════════════════════════════════════════
cat_c = []

subq_questions = [
    ("Quels utilisateurs ont fait plus de SELECT que la moyenne dans l'audit ?",
     "SELECT DB_USER FROM (SELECT DB_USER, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DB_USER) WHERE NB > (SELECT AVG(NB) FROM (SELECT COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DB_USER));"),
    ("Quel utilisateur Oracle a effectué le SELECT le plus récent ?",
     "SELECT DB_USER, OBJ_NAME, SQL_TEXT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND TIMESTAMP = (SELECT MAX(TIMESTAMP) FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT');"),
    ("Quels utilisateurs n'ont jamais fait de SELECT dans l'audit Oracle ?",
     "SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE DB_USER NOT IN (SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT');"),
    ("Quels utilisateurs ont consulté DBA_USERS ou V$SESSION en SELECT ?",
     "SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND OBJ_NAME IN ('DBA_USERS','V$SESSION');"),
    ("Quels utilisateurs n'ont jamais eu d'erreur Oracle en SELECT ?",
     "SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND DB_USER NOT IN (SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND RETURNCODE!=0);"),
    ("Quel jour a eu le plus de SELECT dans l'audit Oracle ?",
     "SELECT SUBSTR(TIMESTAMP,1,10) AS JOUR, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY SUBSTR(TIMESTAMP,1,10) ORDER BY NB DESC FETCH FIRST 1 ROWS ONLY;"),
    ("SELECT de l'utilisateur le plus actif de l'audit Oracle.",
     "SELECT AUDIT_ID, TIMESTAMP, OBJ_NAME, SQL_TEXT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND DB_USER = (SELECT DB_USER FROM ORACLE_AUDIT_TRAIL GROUP BY DB_USER ORDER BY COUNT(*) DESC FETCH FIRST 1 ROWS ONLY);"),
    ("Quels objets n'ont jamais été consultés en SELECT dans l'audit ?",
     "SELECT DISTINCT OBJ_NAME FROM ORACLE_AUDIT_TRAIL WHERE OBJ_NAME NOT IN (SELECT DISTINCT OBJ_NAME FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND OBJ_NAME IS NOT NULL AND OBJ_NAME!='') AND OBJ_NAME IS NOT NULL AND OBJ_NAME!='';"),
    ("Utilisateurs ayant généré des erreurs ORA-01017 et aussi effectué des SELECT.",
     "SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND DB_USER IN (SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE RETURNCODE=1017);"),
    ("SELECT effectués lors de la session Oracle la plus longue.",
     "SELECT AUDIT_ID, TIMESTAMP, DB_USER, OBJ_NAME FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND SESSIONID = (SELECT SESSIONID FROM ORACLE_AUDIT_TRAIL GROUP BY SESSIONID ORDER BY COUNT(*) DESC FETCH FIRST 1 ROWS ONLY);"),
    ("Quels utilisateurs ont une activité SELECT uniquement nocturne (après 20h) ?",
     "SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND DB_USER NOT IN (SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND TO_NUMBER(SUBSTR(TIMESTAMP,12,2)) < 20);"),
    ("Première et dernière requête SELECT de chaque utilisateur Oracle.",
     "SELECT DB_USER, MIN(TIMESTAMP) AS PREMIER_SELECT, MAX(TIMESTAMP) AS DERNIER_SELECT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DB_USER ORDER BY DB_USER;"),
    ("Quels SELECT ont été effectués sur des objets système sensibles ?",
     "SELECT AUDIT_ID, TIMESTAMP, DB_USER, OBJ_NAME FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND OBJ_NAME IN ('DBA_USERS','V$SESSION','ALL_TABLES','DBA_OBJECTS','V$SQL') ORDER BY TIMESTAMP;"),
    ("Y a-t-il des SELECT pendant les heures ouvrées (8h-18h) ?",
     "SELECT COUNT(*) AS NB_SELECT_OUVRE FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND TO_NUMBER(SUBSTR(TIMESTAMP,12,2)) BETWEEN 8 AND 18;"),
    ("Top 5 des utilisateurs les plus actifs en SELECT dans l'audit Oracle.",
     "SELECT DB_USER, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DB_USER ORDER BY NB DESC FETCH FIRST 5 ROWS ONLY;"),
]

for instr, sql in subq_questions:
    cat_c.append({"instruction": instr, "output": sql})
    syn = instr.replace("Quels","Identifier les").replace("Quel","Trouver le")
    if syn != instr:
        cat_c.append({"instruction": syn, "output": sql})
    cat_c.append({"instruction": "Analyse avancée : " + instr.lower(), "output": sql})

base_c = cat_c[:]
while len(cat_c) < 300:
    ex = random.choice(base_c)
    pfx = random.choice(["Détection : ","Vérification : ","Rapport sécurité : "])
    cat_c.append({"instruction": pfx + ex["instruction"].lower().capitalize(), "output": ex["output"]})
cat_c = cat_c[:300]

# ════════════════════════════════════════════════════════════
# CAT D — Sécurité / Anomalies / Oracle avancé   CIBLE : 200
# ════════════════════════════════════════════════════════════
cat_d = []

secu_avance = [
    # Refus
    ("Supprime tous les enregistrements d'audit.",
     "-- REQUÊTE REFUSÉE : les opérations DELETE ne sont pas autorisées. Ce système est en lecture seule (SELECT uniquement)."),
    ("Tronque la table d'audit Oracle.",
     "-- REQUÊTE REFUSÉE : TRUNCATE non autorisé. La table d'audit est immuable."),
    ("Modifie les entrées d'audit Oracle.",
     "-- REQUÊTE REFUSÉE : UPDATE non autorisé. La table d'audit est protégée en lecture seule."),
    ("Insère une fausse entrée dans l'audit Oracle.",
     "-- REQUÊTE REFUSÉE : INSERT non autorisé. La table d'audit est protégée."),
    ("Supprime la table ORACLE_AUDIT_TRAIL.",
     "-- REQUÊTE REFUSÉE : DROP TABLE non autorisé."),
    ("Accorde le rôle DBA à un utilisateur.",
     "-- REQUÊTE REFUSÉE : GRANT non autorisé depuis ce système."),
    # Détection anomalies
    ("Détecte les tentatives de connexion échouées dans l'audit Oracle.",
     "SELECT DB_USER, OS_HOST, COUNT(*) AS NB_ECHECS FROM ORACLE_AUDIT_TRAIL WHERE RETURNCODE=1017 GROUP BY DB_USER, OS_HOST ORDER BY NB_ECHECS DESC;"),
    ("Détecte les SELECT nocturnes suspects dans l'audit (après 22h).",
     "SELECT AUDIT_ID, TIMESTAMP, DB_USER, OBJ_NAME FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND TO_NUMBER(SUBSTR(TIMESTAMP,12,2)) >= 22 ORDER BY TIMESTAMP;"),
    ("Y a-t-il des accès à DBA_USERS hors heures ouvrées ?",
     "SELECT AUDIT_ID, TIMESTAMP, DB_USER FROM ORACLE_AUDIT_TRAIL WHERE OBJ_NAME='DBA_USERS' AND (TO_NUMBER(SUBSTR(TIMESTAMP,12,2)) < 8 OR TO_NUMBER(SUBSTR(TIMESTAMP,12,2)) >= 18) ORDER BY TIMESTAMP;"),
    ("Rapport de sécurité : utilisateurs avec plus de 5 erreurs d'authentification.",
     "SELECT DB_USER, COUNT(*) AS NB_ECHECS FROM ORACLE_AUDIT_TRAIL WHERE RETURNCODE=1017 GROUP BY DB_USER HAVING COUNT(*) > 5 ORDER BY NB_ECHECS DESC;"),
    # Oracle avancé
    ("Affiche les 10 derniers SELECT dans l'audit Oracle.",
     "SELECT AUDIT_ID, TIMESTAMP, DB_USER, OBJ_NAME, SQL_TEXT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' ORDER BY TIMESTAMP DESC FETCH FIRST 10 ROWS ONLY;"),
    ("Rang des utilisateurs Oracle par nombre de SELECT (RANK).",
     "SELECT DB_USER, COUNT(*) AS NB, RANK() OVER (ORDER BY COUNT(*) DESC) AS RANG FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DB_USER;"),
    ("Affiche le pourcentage de SELECT de chaque utilisateur Oracle.",
     "SELECT DB_USER, COUNT(*) AS NB, ROUND(COUNT(*)*100/SUM(COUNT(*)) OVER(),2) AS PCT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DB_USER ORDER BY PCT DESC;"),
    ("Somme cumulative des SELECT Oracle par jour (fenêtre glissante).",
     "SELECT SUBSTR(TIMESTAMP,1,10) AS JOUR, COUNT(*) AS NB_JOUR, SUM(COUNT(*)) OVER (ORDER BY SUBSTR(TIMESTAMP,1,10)) AS CUMUL FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY SUBSTR(TIMESTAMP,1,10) ORDER BY JOUR;"),
    ("Affiche le statut de chaque événement d'audit (succès ou code erreur).",
     "SELECT AUDIT_ID, DB_USER, ACTION_NAME, DECODE(RETURNCODE,0,'Succès','Erreur ORA-'||RETURNCODE) AS STATUT FROM ORACLE_AUDIT_TRAIL;"),
    ("NVL : remplace les commentaires vides par 'Non renseigné' dans l'audit.",
     "SELECT AUDIT_ID, DB_USER, NVL(NULLIF(COMMENT_TEXT,''),'Non renseigné') AS COMMENTAIRE FROM ORACLE_AUDIT_TRAIL;"),
]

for instr, sql in secu_avance:
    cat_d.append({"instruction": instr, "output": sql})
    syn = instr.replace("Détecte","Identifier").replace("Affiche","Montre").replace("Rapport","Bilan")
    if syn != instr:
        cat_d.append({"instruction": syn, "output": sql})

base_d = cat_d[:]
while len(cat_d) < 200:
    ex = random.choice(base_d)
    pfx = random.choice(["Audit sécurité : ","Contrôle Oracle : ","Vérification : "])
    cat_d.append({"instruction": pfx + ex["instruction"].lower().capitalize(), "output": ex["output"]})
cat_d = cat_d[:200]

# ════════════════════════════════════════════════════════════
# CAT E — Questions naturelles variées   CIBLE : 300
# ════════════════════════════════════════════════════════════
cat_e = []

naturel = [
    ("Montre-moi tout ce que SYS a consulté en SELECT dans l'audit Oracle.",
     "SELECT TIMESTAMP, OBJ_NAME, SQL_TEXT FROM ORACLE_AUDIT_TRAIL WHERE DB_USER='SYS' AND ACTION_NAME='SELECT' ORDER BY TIMESTAMP;"),
    ("J'ai besoin de savoir qui a lu la table EMP.",
     "SELECT DISTINCT DB_USER, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE OBJ_NAME='EMP' AND ACTION_NAME='SELECT' GROUP BY DB_USER ORDER BY NB DESC;"),
    ("Est-ce que quelqu'un a consulté DBA_USERS sans autorisation ?",
     "SELECT AUDIT_ID, TIMESTAMP, DB_USER, PRIVILEGE_USED FROM ORACLE_AUDIT_TRAIL WHERE OBJ_NAME='DBA_USERS' AND ACTION_NAME='SELECT' AND (PRIVILEGE_USED IS NULL OR PRIVILEGE_USED='') ORDER BY TIMESTAMP;"),
    ("Donne-moi les 20 derniers événements de l'audit Oracle.",
     "SELECT * FROM ORACLE_AUDIT_TRAIL ORDER BY TIMESTAMP DESC FETCH FIRST 20 ROWS ONLY;"),
    ("Qui s'est connecté à Oracle cette semaine dans l'audit ?",
     "SELECT DISTINCT DB_USER, OS_HOST, MIN(TIMESTAMP) AS PREMIERE_CONNEXION FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON' AND SUBSTR(TIMESTAMP,1,10) >= TO_CHAR(SYSDATE-7,'YYYY-MM-DD') GROUP BY DB_USER, OS_HOST ORDER BY DB_USER;"),
    ("L'utilisateur SCOTT a-t-il consulté des données sensibles ?",
     "SELECT AUDIT_ID, TIMESTAMP, OBJ_NAME, SQL_TEXT FROM ORACLE_AUDIT_TRAIL WHERE DB_USER='SCOTT' AND ACTION_NAME='SELECT' AND OBJ_NAME IN ('DBA_USERS','V$SESSION','PAYROLL','ACCOUNTS') ORDER BY TIMESTAMP;"),
    ("Montre-moi toutes les erreurs enregistrées dans l'audit Oracle.",
     "SELECT AUDIT_ID, TIMESTAMP, DB_USER, ACTION_NAME, RETURNCODE FROM ORACLE_AUDIT_TRAIL WHERE RETURNCODE!=0 ORDER BY TIMESTAMP;"),
    ("Donne-moi un tableau de bord de l'activité Oracle.",
     "SELECT ACTION_NAME, COUNT(*) AS NB, ROUND(COUNT(*)*100/SUM(COUNT(*)) OVER(),1) AS PCT FROM ORACLE_AUDIT_TRAIL GROUP BY ACTION_NAME ORDER BY NB DESC;"),
    ("Quand a eu lieu le premier et le dernier SELECT dans l'audit Oracle ?",
     "SELECT MIN(TIMESTAMP) AS PREMIER_SELECT, MAX(TIMESTAMP) AS DERNIER_SELECT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT';"),
    ("Combien d'utilisateurs distincts ont fait des SELECT dans l'audit ?",
     "SELECT COUNT(DISTINCT DB_USER) AS NB_UTILISATEURS FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT';"),
    ("Rapport d'audit Oracle pour la conformité : actions sensibles.",
     "SELECT DB_USER, ACTION_NAME, OBJ_NAME, TIMESTAMP, RETURNCODE, PRIVILEGE_USED FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME IN ('GRANT','REVOKE','CREATE TABLE','DROP TABLE','ALTER TABLE') ORDER BY TIMESTAMP;"),
    ("Est-ce que HR a fait des SELECT sur des tables de paie ?",
     "SELECT AUDIT_ID, TIMESTAMP, OBJ_NAME, SQL_TEXT FROM ORACLE_AUDIT_TRAIL WHERE DB_USER='HR' AND ACTION_NAME='SELECT' AND OBJ_NAME IN ('PAYROLL','ACCOUNTS','BUDGET') ORDER BY TIMESTAMP;"),
    ("Recherche toute activité suspecte dans l'audit Oracle.",
     "SELECT AUDIT_ID, TIMESTAMP, DB_USER, ACTION_NAME, OBJ_NAME, RETURNCODE FROM ORACLE_AUDIT_TRAIL WHERE RETURNCODE!=0 OR TO_NUMBER(SUBSTR(TIMESTAMP,12,2))>=22 OR ACTION_NAME IN ('DROP TABLE','TRUNCATE TABLE','REVOKE') ORDER BY TIMESTAMP DESC;"),
    ("Quel est le volume de SELECT Oracle par mois dans l'audit ?",
     "SELECT SUBSTR(TIMESTAMP,1,7) AS MOIS, COUNT(*) AS NB_SELECT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY SUBSTR(TIMESTAMP,1,7) ORDER BY MOIS;"),
    ("Analyse l'audit Oracle pour identifier les comptes les plus actifs en SELECT.",
     "SELECT DB_USER, COUNT(*) AS NB_SELECT, COUNT(DISTINCT OBJ_NAME) AS NB_OBJETS, COUNT(DISTINCT OS_HOST) AS NB_SERVEURS FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DB_USER ORDER BY NB_SELECT DESC;"),
]

for instr, sql in naturel:
    cat_e.append({"instruction": instr, "output": sql})
    for pfx in ["Question métier : ","Besoin de : ","Analyse : "]:
        if len(cat_e) < 300:
            cat_e.append({"instruction": pfx + instr.lower().capitalize(), "output": sql})

base_e = cat_e[:]
while len(cat_e) < 300:
    ex = random.choice(base_e)
    cat_e.append({"instruction": "Requête audit Oracle : " + ex["instruction"].lower().capitalize(), "output": ex["output"]})
cat_e = cat_e[:300]

# ════════════════════════════════════════════════════════════
# SQL → FR : 2000 exemples enrichis avec focus SELECT
# ════════════════════════════════════════════════════════════
sql_fr_base = [
    ("SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' ORDER BY DB_USER;",
     "Cette requête retourne la liste des utilisateurs Oracle ayant effectué au moins une requête SELECT enregistrée dans l'audit."),
    ("SELECT COUNT(*) AS NB_SELECT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT';",
     "Cette requête compte le nombre total de requêtes SELECT enregistrées dans la table d'audit Oracle."),
    ("SELECT DB_USER, COUNT(*) AS NB_SELECT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DB_USER ORDER BY NB_SELECT DESC;",
     "Cette requête affiche le nombre de requêtes SELECT effectuées par chaque utilisateur Oracle, du plus actif au moins actif."),
    ("SELECT DISTINCT OBJ_NAME FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND OBJ_NAME IS NOT NULL AND OBJ_NAME!='';",
     "Cette requête retourne la liste des objets Oracle (tables, vues) qui ont été consultés via des requêtes SELECT enregistrées dans l'audit."),
    ("SELECT AUDIT_ID, TIMESTAMP, DB_USER, OBJ_NAME, SQL_TEXT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' ORDER BY TIMESTAMP DESC FETCH FIRST 10 ROWS ONLY;",
     "Cette requête retourne les 10 requêtes SELECT les plus récentes enregistrées dans l'audit Oracle, avec le texte SQL complet."),
    ("SELECT COUNT(*) AS NB_CONNEXIONS FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON';",
     "Cette requête compte le nombre total de connexions (LOGON) enregistrées dans l'audit Oracle."),
    ("SELECT DB_USER, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY DB_USER ORDER BY NB DESC;",
     "Cette requête affiche le volume total d'activité par utilisateur Oracle dans l'audit, du plus actif au moins actif."),
    ("SELECT AUDIT_ID, TIMESTAMP, DB_USER, ACTION_NAME, RETURNCODE FROM ORACLE_AUDIT_TRAIL WHERE RETURNCODE!=0;",
     "Cette requête retourne tous les événements d'audit Oracle ayant généré une erreur (code retour différent de zéro)."),
    ("SELECT DISTINCT DB_USER, OS_HOST FROM ORACLE_AUDIT_TRAIL WHERE RETURNCODE=1017;",
     "Cette requête identifie les utilisateurs et serveurs ayant généré des erreurs ORA-01017, c'est-à-dire des tentatives de connexion avec un mot de passe incorrect."),
    ("SELECT OBJ_NAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY OBJ_NAME ORDER BY NB DESC FETCH FIRST 10 ROWS ONLY;",
     "Cette requête retourne les 10 objets Oracle les plus consultés en SELECT dans l'audit, permettant d'identifier les tables les plus sollicitées."),
    ("SELECT SUBSTR(TIMESTAMP,1,10) AS JOUR, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY SUBSTR(TIMESTAMP,1,10) ORDER BY JOUR;",
     "Cette requête calcule le nombre de requêtes SELECT par jour dans l'audit Oracle, utile pour visualiser l'évolution de l'activité de consultation."),
    ("SELECT DB_USER, COUNT(*) AS NB, RANK() OVER (ORDER BY COUNT(*) DESC) AS RANG FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DB_USER;",
     "Cette requête classe les utilisateurs Oracle par nombre de SELECT effectués, en leur attribuant un rang : le rang 1 correspond à l'utilisateur ayant fait le plus de requêtes."),
    ("SELECT COUNT(*) AS TOTAL, SUM(CASE WHEN RETURNCODE=0 THEN 1 ELSE 0 END) AS SUCCES, SUM(CASE WHEN RETURNCODE!=0 THEN 1 ELSE 0 END) AS ERREURS FROM ORACLE_AUDIT_TRAIL;",
     "Cette requête fournit un bilan global de l'audit Oracle : le nombre total d'événements, le nombre de succès et le nombre d'erreurs."),
    ("SELECT DB_USER, COUNT(DISTINCT OBJ_NAME) AS NB_OBJETS FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DB_USER ORDER BY NB_OBJETS DESC;",
     "Cette requête mesure la diversité des consultations de chaque utilisateur Oracle en comptant le nombre d'objets distincts qu'il a lus en SELECT."),
    ("SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND OBJ_NAME IN ('DBA_USERS','V$SESSION');",
     "Cette requête identifie les utilisateurs Oracle ayant consulté des vues système sensibles (DBA_USERS et V$SESSION) via des SELECT."),
    ("SELECT TO_NUMBER(SUBSTR(TIMESTAMP,12,2)) AS HEURE, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY TO_NUMBER(SUBSTR(TIMESTAMP,12,2)) ORDER BY HEURE;",
     "Cette requête analyse la répartition des requêtes SELECT Oracle par heure de la journée, utile pour détecter des accès nocturnes anormaux."),
    ("SELECT ROUND(SUM(CASE WHEN RETURNCODE=0 THEN 1 ELSE 0 END)*100/COUNT(*),2) AS TAUX_SUCCES FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT';",
     "Cette requête calcule le taux de succès des requêtes SELECT dans l'audit Oracle, exprimé en pourcentage."),
    ("SELECT DB_USER, MIN(TIMESTAMP) AS PREMIER_SELECT, MAX(TIMESTAMP) AS DERNIER_SELECT FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' GROUP BY DB_USER ORDER BY DB_USER;",
     "Cette requête retourne pour chaque utilisateur Oracle la date de sa première et de sa dernière requête SELECT enregistrée dans l'audit."),
    ("SELECT AUDIT_ID, TIMESTAMP, DB_USER, OBJ_NAME FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' AND TO_NUMBER(SUBSTR(TIMESTAMP,12,2)) >= 22 ORDER BY TIMESTAMP;",
     "Cette requête identifie les requêtes SELECT Oracle effectuées après 22h, ce qui constitue une activité nocturne potentiellement suspecte."),
    ("SELECT ACTION_NAME, COUNT(*) AS NB, ROUND(COUNT(*)*100/SUM(COUNT(*)) OVER(),1) AS PCT FROM ORACLE_AUDIT_TRAIL GROUP BY ACTION_NAME ORDER BY NB DESC;",
     "Cette requête affiche la distribution de toutes les actions Oracle dans l'audit avec leur pourcentage, permettant de voir que SELECT domine largement l'activité."),
]

sqlplus_results = [
    ("DB_USER    NB_SELECT\nSCOTT           45\nHR              38\nAPP_USER        27\nSYS             19\nREPORT_USR      15",
     "Le rapport d'activité SELECT Oracle montre que SCOTT est l'utilisateur le plus actif en consultation avec 45 requêtes SELECT, suivi de HR (38), APP_USER (27), SYS (19) et REPORT_USR (15)."),
    ("NB_SELECT\n---------\n347",
     "L'audit Oracle enregistre 347 requêtes SELECT au total sur la période analysée. C'est la principale activité de la base de données, ce qui est normal."),
    ("OBJ_NAME    NB\nEMP         42\nDEPT        28\nPAYROLL     19\nACCOUNTS    15\nDBA_USERS    8",
     "Les objets Oracle les plus consultés en SELECT sont EMP (42 accès), DEPT (28) et PAYROLL (19). Les accès à DBA_USERS (8) méritent une attention particulière car c'est une vue système sensible."),
    ("AUDIT_ID  TIMESTAMP            DB_USER  OBJ_NAME   RETURNCODE\n1234      2026-02-15 23:42:11  SCOTT    DBA_USERS  0",
     "Une requête SELECT suspecte a été détectée : SCOTT a consulté DBA_USERS le 15 février 2026 à 23h42, soit en pleine nuit. Cet accès nocturne à une vue système sensible doit être vérifié."),
    ("TOTAL  SUCCES  ERREURS  TAUX_SUCCES\n500    468     32       93.6%",
     "Le bilan global de l'audit Oracle montre 500 événements, dont 468 succès (93.6%) et 32 erreurs (6.4%). Le taux de succès est satisfaisant mais les 32 erreurs méritent une analyse."),
    ("HEURE  NB\n8       45\n9       52\n10      48\n14      41\n15      38\n22       8\n23       5",
     "L'analyse horaire des SELECT Oracle montre un pic d'activité en matinée (8h-10h) avec 45-52 requêtes par heure, et une activité normale en après-midi. Les 13 requêtes nocturnes (22h-23h) sont à surveiller."),
    ("DB_USER    NB_ECHECS_AUTH\nSCOTT           7\nHR              3\nAPP_USER         2",
     "L'audit Oracle détecte des échecs d'authentification ORA-01017 : SCOTT a échoué 7 fois, HR 3 fois et APP_USER 2 fois. Les 7 tentatives de SCOTT sont préoccupantes et suggèrent une possible tentative d'intrusion."),
    ("MOIS      NB_SELECT\n2026-01       198\n2026-02       149",
     "L'évolution des SELECT Oracle montre 198 requêtes en janvier 2026 et 149 en février (mois incomplet). L'activité de consultation est régulière et cohérente avec l'usage normal d'une base de données d'entreprise."),
]

prefixes = [
    "Explique cette requête d'audit Oracle : ",
    "Que retourne cette requête sur ORACLE_AUDIT_TRAIL ? ",
    "Décris en français ce que fait : ",
    "Traduis ce résultat d'audit Oracle en français : ",
    "Reformule en langage naturel pour un non-technicien : ",
    "Qu'est-ce que cette requête d'audit Oracle retourne ? ",
    "En français clair, que signifie ce résultat Oracle ? ",
    "Explique pour le responsable sécurité : ",
]
result_prefixes = [
    "Voici un résultat SQL*Plus Oracle d'audit, explique-le en français :\n",
    "Traduis ce résultat brut d'audit Oracle en langage naturel :\n",
    "Un manager demande ce que signifie ce résultat Oracle :\n",
    "Reformule ce résultat d'exécution SQL Oracle en phrase compréhensible :\n",
    "Explique ce résultat d'audit Oracle pour un rapport de conformité :\n",
]

sql_fr_examples = []
for sql, explication in sql_fr_base:
    for prefix in prefixes:
        sql_fr_examples.append({"instruction": prefix + sql, "output": explication})

for result, explanation in sqlplus_results:
    for rp in result_prefixes:
        sql_fr_examples.append({"instruction": rp + result, "output": explanation})

var_replacements = [
    ("Cette requête retourne","Le résultat de cette requête est"),
    ("Cette requête affiche","Cette commande SQL affiche"),
    ("Cette requête calcule","Cette instruction Oracle calcule"),
    ("Cette requête compte","Cette requête permet de compter"),
    ("Cette requête identifie","Cette analyse de sécurité identifie"),
    ("Cette requête mesure","Cette analyse Oracle mesure"),
    ("Cette requête classe","Ce rapport Oracle classe"),
    ("Cette requête analyse","Cette instruction Oracle analyse"),
]
while len(sql_fr_examples) < 2000:
    sql, explication = random.choice(sql_fr_base)
    prefix = random.choice(prefixes)
    old, new = random.choice(var_replacements)
    variant = explication.replace(old, new)
    sql_fr_examples.append({"instruction": prefix + sql, "output": variant})
sql_fr_examples = sql_fr_examples[:2000]

# ════════════════════════════════════════════════════════════
# ASSEMBLAGE & STATS
# ════════════════════════════════════════════════════════════
fr_sql_all = cat_a + cat_b + cat_c + cat_d + cat_e
random.shuffle(fr_sql_all)
random.shuffle(sql_fr_examples)
all_examples = fr_sql_all + sql_fr_examples
random.shuffle(all_examples)

df = pd.DataFrame(all_examples)
df.to_csv("oracle_nlp_dataset.csv", index=False)

print("="*60)
print("  DATASET FINAL — DISTRIBUTION ORIENTÉE SELECT")
print("="*60)
print(f"  FR→SQL   : {len(fr_sql_all):>5} exemples")
print(f"  SQL→FR   : {len(sql_fr_examples):>5} exemples")
print(f"  TOTAL    : {len(all_examples):>5} exemples")
print()
cats = [("SELECT / Filtres simples", cat_a), ("Agrégations GROUP BY/HAVING", cat_b),
        ("Sous-requêtes avancées", cat_c), ("Sécurité / Oracle avancé", cat_d),
        ("Questions naturelles variées", cat_e)]
print("  Distribution FR→SQL :")
for name, cat in cats:
    pct = len(cat)/len(fr_sql_all)*100
    bar = "█" * (len(cat) // 25)
    print(f"    {name:<35} {len(cat):>4}  ({pct:>4.1f}%)  {bar}")
print()
# Compter les SELECT dans les outputs
nb_select = sum(1 for ex in fr_sql_all
                if "ACTION_NAME='SELECT'" in ex["output"] or
                   "ACTION_NAME=\\'SELECT\\'" in ex["output"])
print(f"  Exemples ciblant SELECT dans FR→SQL : {nb_select} ({nb_select/len(fr_sql_all)*100:.0f}%)")
print("="*60)
print("✅ Dataset équilibré, centré SELECT, prêt pour l'entraînement.")

  DATASET FINAL — DISTRIBUTION ORIENTÉE SELECT
  FR→SQL   :  2000 exemples
  SQL→FR   :  2000 exemples
  TOTAL    :  4000 exemples

  Distribution FR→SQL :
    SELECT / Filtres simples             700  (35.0%)  ████████████████████████████
    Agrégations GROUP BY/HAVING          500  (25.0%)  ████████████████████
    Sous-requêtes avancées               300  (15.0%)  ████████████
    Sécurité / Oracle avancé             200  (10.0%)  ████████
    Questions naturelles variées         300  (15.0%)  ████████████

  Exemples ciblant SELECT dans FR→SQL : 1432 (72%)
✅ Dataset équilibré, centré SELECT, prêt pour l'entraînement.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 6 — Dataset ultra-varié et contextuel             ║
# ║  Génération avancée : schéma, COUNT DISTINCT, temporalité  ║
# ║  Validation/correction, mapping entité, cas complexes      ║
# ╚══════════════════════════════════════════════════════════════╝
import itertools

# Schéma simulé pour contextualisation
SCHEMA = {
    "EMP": ["ID", "NOM", "DEPT", "SALAIRE", "DATE_EMBAUCHE"],
    "CLIENT": ["ID", "NOM", "VILLE", "DATE_CREATION"],
    "ORDERS": ["ORDER_ID", "CLIENT_ID", "DATE", "MONTANT", "STATUS"],
    "DBA_USERS": ["USERNAME", "CREATED", "PROFILE", "ACCOUNT_STATUS"],
    "AUDIT_LOG": ["AUDIT_ID", "TIMESTAMP", "DB_USER", "ACTION_NAME", "OBJ_NAME", "RETURNCODE"]
}

# Générateur de contexte schéma
schema_context = "Schéma : " + ", ".join([
    f"{table}({', '.join(cols)})" for table, cols in SCHEMA.items()
])

# Exemples complexes et variés
ultra_examples = []

# COUNT DISTINCT, agrégats avancés
ultra_examples.append({
    "instruction": f"{schema_context} — Combien d'utilisateurs DISTINCTS ont modifié la table ORDERS en février 2026 ?",
    "output": "SELECT COUNT(DISTINCT DB_USER) AS NB_UTILS FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME IN ('UPDATE','INSERT','DELETE') AND OBJ_NAME='ORDERS' AND TIMESTAMP LIKE '2026-02%';"
})
ultra_examples.append({
    "instruction": f"{schema_context} — Quels sont les 3 utilisateurs ayant modifié le plus de tables différentes la semaine dernière ?",
    "output": "SELECT DB_USER, COUNT(DISTINCT OBJ_NAME) AS NB_TABLES FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME IN ('UPDATE','INSERT','DELETE') AND TIMESTAMP >= TO_CHAR(SYSDATE-7,'YYYY-MM-DD') GROUP BY DB_USER ORDER BY NB_TABLES DESC FETCH FIRST 3 ROWS ONLY;"
})
# Temporalité complexe
ultra_examples.append({
    "instruction": f"{schema_context} — Donne-moi les 5 dernières actions de chaque utilisateur sur la table CLIENT.",
    "output": "SELECT * FROM (SELECT DB_USER, ACTION_NAME, OBJ_NAME, TIMESTAMP, ROW_NUMBER() OVER (PARTITION BY DB_USER ORDER BY TIMESTAMP DESC) AS RN FROM ORACLE_AUDIT_TRAIL WHERE OBJ_NAME='CLIENT') WHERE RN <= 5;"
})
# Multi-actions et jointures
ultra_examples.append({
    "instruction": f"{schema_context} — Liste les utilisateurs ayant fait des SELECT, UPDATE et DELETE sur la même table.",
    "output": "SELECT DB_USER, OBJ_NAME FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME IN ('SELECT','UPDATE','DELETE') GROUP BY DB_USER, OBJ_NAME HAVING COUNT(DISTINCT ACTION_NAME)=3;"
})
# Cas d'erreur et sécurité
ultra_examples.append({
    "instruction": f"{schema_context} — Quels utilisateurs ont généré des erreurs sur plusieurs tables ?",
    "output": "SELECT DB_USER, COUNT(DISTINCT OBJ_NAME) AS NB_TABLES FROM ORACLE_AUDIT_TRAIL WHERE RETURNCODE!=0 GROUP BY DB_USER HAVING COUNT(DISTINCT OBJ_NAME)>1;"
})
# Questions ambiguës
ultra_examples.append({
    "instruction": f"{schema_context} — Qui sont les utilisateurs actifs ?",
    "output": "SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME IN ('SELECT','UPDATE','INSERT') AND TIMESTAMP >= TO_CHAR(SYSDATE-30,'YYYY-MM-DD');"
})
# Correction automatique (exemple)
ultra_examples.append({
    "instruction": f"{schema_context} — Donne-moi le nombre d'actions sur la table EMP.",
    "output": "SELECT COUNT(*) AS NB_ACTIONS FROM ORACLE_AUDIT_TRAIL WHERE OBJ_NAME='EMP';"
})
ultra_examples.append({
    "instruction": f"{schema_context} — Donne-moi le nombre d'actions DISTINCTES sur la table EMP.",
    "output": "SELECT COUNT(DISTINCT ACTION_NAME) AS NB_ACTIONS_DISTINCT FROM ORACLE_AUDIT_TRAIL WHERE OBJ_NAME='EMP';"
})
# Génération automatique de paraphrases
paraphrases = [
    ("Combien de", "Quel est le nombre de"),
    ("Donne-moi", "Affiche-moi"),
    ("Liste", "Montre-moi"),
    ("Quels", "Quelles sont les"),
    ("Qui sont", "Quels sont")
]
for ex in ultra_examples[:]:
    for old, new in paraphrases:
        if old in ex["instruction"]:
            ultra_examples.append({
                "instruction": ex["instruction"].replace(old, new, 1),
                "output": ex["output"]
            })

# Validation/correction automatique (exemple simple)
def validate_sql(sql):
    if "SELECT *" in sql:
        return sql.replace("SELECT *", "SELECT DB_USER, ACTION_NAME, OBJ_NAME, TIMESTAMP")
    return sql
for ex in ultra_examples:
    ex["output"] = validate_sql(ex["output"])

# Ajout au dataset principal
all_examples += ultra_examples
random.shuffle(all_examples)
df = pd.DataFrame(all_examples)
df.to_csv("oracle_nlp_dataset_ultra.csv", index=False)
print("✅ Dataset ultra-varié et contextuel généré : oracle_nlp_dataset_ultra.csv")


In [5]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 6 — Fine-tuning LoRA sur ORACLE_AUDIT_TRAIL         ║
# ╚══════════════════════════════════════════════════════════════╝
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
import torch, shutil, os

LORA_DIR = "tinyllama_oracle_lora"

# ── Tokenizer ────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ── Modèle de base ───────────────────────────────────────────
print("📥 Chargement du modèle de base...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, torch_dtype=torch.float16, device_map="auto"
)

# ── Préparation + LoRA (ordre correct) ───────────────────────
base_model = prepare_model_for_kbit_training(base_model)
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj","v_proj","k_proj","o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(base_model, lora_config)
print("  Paramètres entraînables :")
model.print_trainable_parameters()

# ── Dataset ──────────────────────────────────────────────────
raw = load_dataset("csv", data_files="oracle_nlp_dataset.csv")["train"]
print(f"  Exemples chargés : {len(raw)}")

def preprocess(examples):
    full_texts = [
        f"[INST] Tu es un expert Oracle Database. Réponds en SQL Oracle valide ou en français naturel.\n{instr}\n[/INST] {out}"
        for instr, out in zip(examples["instruction"], examples["output"])
    ]
    tok = tokenizer(full_texts, truncation=True, max_length=384, padding="max_length")
    tok["labels"] = tok["input_ids"].copy()
    return tok

tokenized = raw.map(preprocess, batched=True, remove_columns=["instruction","output"])
tokenized.set_format(type="torch", columns=["input_ids","attention_mask","labels"])

# ── Entraînement ─────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=LORA_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=50,
    save_strategy="epoch",
    report_to=[],
    push_to_hub=False,
    optim="adamw_torch",
    dataloader_num_workers=2,
)

trainer = Trainer(model=model, args=training_args, train_dataset=tokenized)
print("🚀 Entraînement en cours...")
trainer.train()

model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"✅ Adapter LoRA sauvegardé : {LORA_DIR}/")
shutil.make_archive(LORA_DIR, "zip", LORA_DIR)
print(f"📦 Archive : {LORA_DIR}.zip")

`torch_dtype` is deprecated! Use `dtype` instead!


📥 Chargement du modèle de base...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Paramètres entraînables :
trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


Generating train split: 0 examples [00:00, ? examples/s]

...

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


🚀 Entraînement en cours...


Step,Training Loss
50,4.740777
100,0.190854
150,0.087077
200,0.056564
250,0.055311
300,0.044432
350,0.040528
400,0.038379
450,0.031775
500,0.030530


Step,Training Loss
50,4.740777
100,0.190854
150,0.087077
200,0.056564
250,0.055311
300,0.044432
350,0.040528
400,0.038379
450,0.031775
500,0.030530


✅ Adapter LoRA sauvegardé : tinyllama_oracle_lora/
📦 Archive : tinyllama_oracle_lora.zip


In [6]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 7 — Chargement du modèle LoRA pour les tests        ║
# ╚══════════════════════════════════════════════════════════════╝
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

LORA_DIR = "tinyllama_oracle_lora"

print(f"📥 Chargement modèle de base : {MODEL_DIR}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, torch_dtype=torch.float16, device_map="auto"
)
print(f"📥 Chargement adapter LoRA : {LORA_DIR}")
model = PeftModel.from_pretrained(base_model, LORA_DIR)
model.eval()

def generate(prompt: str, max_new_tokens: int = 150, temperature: float = 0.2) -> str:
    """Génère une réponse à partir d'un prompt."""
    full_prompt = (
        f"[INST] Tu es un expert Oracle Database. "
        f"Réponds en SQL Oracle valide ou en français naturel.\n{prompt}\n[/INST]"
    )
    inputs = tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=True,
            pad_token_id=tokenizer.eos_token_id, repetition_penalty=1.1
        )
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)
    return decoded.split("[/INST]")[-1].strip()

print("✅ Modèle prêt pour les tests interactifs.")
print(f"   Device : {next(model.parameters()).device}")

📥 Chargement modèle de base : TinyLlama-1.1B-Chat-v1.0


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

📥 Chargement adapter LoRA : tinyllama_oracle_lora
✅ Modèle prêt pour les tests interactifs.
   Device : cuda:0


In [7]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 8 — Aperçu détaillé de la table ORACLE_AUDIT_TRAIL  ║
# ║  À lire avant de poser vos questions de test                  ║
# ╚══════════════════════════════════════════════════════════════╝
import pandas as pd

AUDIT_DF = pd.read_csv("oracle_audit_trail.csv")

print("╔══════════════════════════════════════════════════════════╗")
print("║         ORACLE_AUDIT_TRAIL — RÉFÉRENCE COMPLÈTE          ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Lignes : {len(AUDIT_DF):<5}  |  Colonnes : {len(AUDIT_DF.columns):<5}               ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  COLONNES INTERROGEABLES :                                ║")
cols_info = [
    ("AUDIT_ID",       "INT",    "Clé primaire auto-incrémentée"),
    ("TIMESTAMP",      "CHAR",   "Format YYYY-MM-DD HH:MM:SS"),
    ("OS_USER",        "CHAR",   "Utilisateur Linux/OS"),
    ("DB_USER",        "CHAR",   "Utilisateur Oracle"),
    ("OS_HOST",        "CHAR",   "Serveur source"),
    ("TERMINAL",       "CHAR",   "Terminal (pts/0, console…)"),
    ("SESSIONID",      "INT",    "ID session Oracle"),
    ("ACTION_NAME",    "CHAR",   "SELECT/LOGON/UPDATE/…"),
    ("OBJ_SCHEMA",     "CHAR",   "Schéma Oracle"),
    ("OBJ_NAME",       "CHAR",   "Nom table/vue"),
    ("SQL_TEXT",       "CHAR",   "Requête SQL exécutée"),
    ("RETURNCODE",     "INT",    "0=OK, >0=erreur ORA-"),
    ("PRIVILEGE_USED", "CHAR",   "Privilège Oracle utilisé"),
    ("STATEMENT",      "INT",    "Numéro de statement"),
    ("COMMENT_TEXT",   "CHAR",   "Commentaire d'audit"),
]
for col, typ, desc in cols_info:
    print(f"║    {col:<20} {typ:<6} {desc:<28}║")
print("╠══════════════════════════════════════════════════════════╣")

print("║  VALEURS DISTINCTES PAR COLONNE CLEF :                   ║")
print(f"║    ACTION_NAME  : {sorted(AUDIT_DF.ACTION_NAME.unique())[:5]}…")
print(f"║    DB_USER      : {sorted(AUDIT_DF.DB_USER.unique())[:5]}…")
print(f"║    OS_HOST      : {sorted(AUDIT_DF.OS_HOST.unique())}")
print(f"║    RETURNCODE   : {sorted(AUDIT_DF.RETURNCODE.unique())}")
print(f"║    OBJ_NAME (top 5) : {list(AUDIT_DF.OBJ_NAME.value_counts().head(5).index)}")

print("╠══════════════════════════════════════════════════════════╣")
print("║  EXEMPLES DE QUESTIONS POSSIBLES :                       ║")
questions_exemples = [
    "Quels utilisateurs Oracle ont effectué des SELECT ?",
    "Combien de connexions LOGON ont été enregistrées ?",
    "Quels événements ont un code retour d'erreur (!=0) ?",
    "Quels serveurs ont le plus d'activité d'audit ?",
    "Activité nocturne suspecte (après 22h) ?",
    "Quels utilisateurs ont accédé à DBA_USERS ?",
    "Résumé global de l'audit Oracle.",
    "Qui a effectué des opérations DELETE ou UPDATE ?",
]
for i, q in enumerate(questions_exemples, 1):
    print(f"║    {i}. {q:<50}║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  5 LIGNES D'APERÇU :                                     ║")
print("╚══════════════════════════════════════════════════════════╝")
print(AUDIT_DF[["AUDIT_ID","TIMESTAMP","DB_USER","ACTION_NAME",
                "OBJ_NAME","RETURNCODE"]].head(5).to_string(index=False))

╔══════════════════════════════════════════════════════════╗
║         ORACLE_AUDIT_TRAIL — RÉFÉRENCE COMPLÈTE          ║
╠══════════════════════════════════════════════════════════╣
║  Lignes : 500    |  Colonnes : 15                  ║
╠══════════════════════════════════════════════════════════╣
║  COLONNES INTERROGEABLES :                                ║
║    AUDIT_ID             INT    Clé primaire auto-incrémentée║
║    TIMESTAMP            CHAR   Format YYYY-MM-DD HH:MM:SS  ║
║    OS_USER              CHAR   Utilisateur Linux/OS        ║
║    DB_USER              CHAR   Utilisateur Oracle          ║
║    OS_HOST              CHAR   Serveur source              ║
║    TERMINAL             CHAR   Terminal (pts/0, console…)  ║
║    SESSIONID            INT    ID session Oracle           ║
║    ACTION_NAME          CHAR   SELECT/LOGON/UPDATE/…       ║
║    OBJ_SCHEMA           CHAR   Schéma Oracle               ║
║    OBJ_NAME             CHAR   Nom table/vue               ║
║    SQL

In [15]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 9 — TEST INTERACTIF : Français → SQL Oracle          ║
# ║  Posez vos propres questions en français                      ║
# ╚══════════════════════════════════════════════════════════════╝
import re

# ── Questions de référence avec SQL attendu ───────────────────
REFERENCE_TESTS = [
    {
        "question": "Quels utilisateurs Oracle ont effectué des SELECT dans l'audit ?",
        "expected_keywords": ["SELECT", "DB_USER", "ACTION_NAME", "SELECT"],
        "expected_sql_fragment": "SELECT",
        "difficulty": "Facile"
    },
    {
        "question": "Combien de connexions LOGON ont été enregistrées dans l'audit ?",
        "expected_keywords": ["COUNT", "LOGON", "ACTION_NAME"],
        "expected_sql_fragment": "COUNT",
        "difficulty": "Facile"
    },
    {
        "question": "Quels événements d'audit se sont soldés par une erreur Oracle ?",
        "expected_keywords": ["RETURNCODE", "0"],
        "expected_sql_fragment": "RETURNCODE",
        "difficulty": "Moyen"
    },
    {
        "question": "Quels utilisateurs Oracle ont généré le plus d'événements d'audit ?",
        "expected_keywords": ["COUNT", "GROUP BY", "DB_USER", "ORDER BY"],
        "expected_sql_fragment": "GROUP BY",
        "difficulty": "Moyen"
    },
    {
        "question": "Y a-t-il de l'activité nocturne suspecte après 22h dans l'audit ?",
        "expected_keywords": ["22", "TIMESTAMP", "SUBSTR"],
        "expected_sql_fragment": "22",
        "difficulty": "Difficile"
    },
    {
        "question": "Quels utilisateurs ont accédé à des vues système sensibles comme DBA_USERS ?",
        "expected_keywords": ["DBA_USERS", "OBJ_NAME"],
        "expected_sql_fragment": "DBA_USERS",
        "difficulty": "Difficile"
    },
    {
        "question": "Résumé global de l'audit Oracle : total événements, utilisateurs, erreurs.",
        "expected_keywords": ["COUNT", "RETURNCODE", "DB_USER"],
        "expected_sql_fragment": "COUNT",
        "difficulty": "Difficile"
    },
    {
        "question": "Supprime tous les enregistrements d'audit.",
        "expected_keywords": ["REFUSÉE", "DELETE", "lecture seule"],
        "expected_sql_fragment": "REFUSÉE",
        "difficulty": "Sécurité"
    },
]

def score_response(response: str, expected_keywords: list, difficulty: str) -> dict:
    """Calcule un score de précision basé sur les mots-clés attendus."""
    resp_upper = response.upper()
    matched = [kw for kw in expected_keywords if kw.upper() in resp_upper]
    score = len(matched) / len(expected_keywords) * 100
    return {
        "score": round(score, 1),
        "matched": matched,
        "missing": [kw for kw in expected_keywords if kw.upper() not in resp_upper],
        "difficulty": difficulty,
    }

def is_valid_sql(text: str) -> bool:
    """Vérifie si la réponse ressemble à du SQL valide ou à un refus."""
    t = text.upper().strip()
    return (t.startswith("SELECT") or t.startswith("--") or
            "FROM" in t or "REFUSÉE" in t)

# ── Tests automatiques sur les questions de référence ─────────
print("="*65)
print("  TEST AUTOMATIQUE — FR → SQL Oracle")
print("  Table testée : ORACLE_AUDIT_TRAIL")
print("="*65)

results = []
total_score = 0

for i, test in enumerate(REFERENCE_TESTS, 1):
    print(f"\n[Test {i}/{len(REFERENCE_TESTS)}] [{test['difficulty']}]")
    print(f"  ❓ Question : {test['question']}")

    response = generate(test["question"], max_new_tokens=120, temperature=0.1)
    scoring  = score_response(response, test["expected_keywords"], test["difficulty"])

    sql_ok = is_valid_sql(response)
    emoji  = "✅" if scoring["score"] >= 70 else ("⚠️" if scoring["score"] >= 40 else "❌")

    print(f"  🔷 SQL généré  : {response[:120]}{'...' if len(response)>120 else ''}")
    print(f"  {emoji} Score     : {scoring['score']}%  |  SQL valide : {'Oui' if sql_ok else 'Non'}")
    if scoring["missing"]:
        print(f"  ⚠️  Manquants  : {scoring['missing']}")

    results.append({"question": test["question"], "response": response,
                    "score": scoring["score"], "difficulty": test["difficulty"],
                    "sql_valid": sql_ok})
    total_score += scoring["score"]

avg = total_score / len(REFERENCE_TESTS)
print("\n" + "="*65)
print("  BILAN DES TESTS FR→SQL")
print("="*65)
by_diff = {}
for r in results:
    by_diff.setdefault(r["difficulty"], []).append(r["score"])
for diff, scores in sorted(by_diff.items()):
    avg_d = sum(scores)/len(scores)
    bar = "█" * int(avg_d/10)
    print(f"  {diff:<12} : {avg_d:>5.1f}%  {bar}")
print(f"  {'─'*40}")
perf_label = "🏆 Excellent" if avg>=80 else ("✅ Bon" if avg>=60 else ("⚠️ Moyen" if avg>=40 else "❌ À améliorer"))
print(f"  Score global : {avg:.1f}%  {perf_label}")
print("="*65)

# ── Zone de test libre ────────────────────────────────────────
print("\n📝 ZONE DE TEST LIBRE — Posez votre propre question :")
print("   Modifiez la variable 'ma_question' ci-dessous et réexécutez.\n")
ma_question = "quel est le dernier utilisatur a modifier la base ?"

print(f"  ❓ Votre question : {ma_question}")
reponse_libre = generate(ma_question, max_new_tokens=150, temperature=0.2)
print(f"  🔷 SQL généré    : {reponse_libre}")
sql_libre_ok = is_valid_sql(reponse_libre)
print(f"  📊 SQL valide    : {'Oui ✅' if sql_libre_ok else 'Non ❌'}")

  TEST AUTOMATIQUE — FR → SQL Oracle
  Table testée : ORACLE_AUDIT_TRAIL

[Test 1/8] [Facile]
  ❓ Question : Quels utilisateurs Oracle ont effectué des SELECT dans l'audit ?
  🔷 SQL généré  : SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='SELECT' ORDER BY DB_USER;
  ✅ Score     : 100.0%  |  SQL valide : Oui

[Test 2/8] [Facile]
  ❓ Question : Combien de connexions LOGON ont été enregistrées dans l'audit ?
  🔷 SQL généré  : SELECT COUNT(*) AS NB_CONNEXIONS FROM ORACLE_AUDIT_TRAIL WHERE ACTION_NAME='LOGON';
  ✅ Score     : 100.0%  |  SQL valide : Oui

[Test 3/8] [Moyen]
  ❓ Question : Quels événements d'audit se sont soldés par une erreur Oracle ?
  🔷 SQL généré  : SELECT AUDIT_ID, TIMESTAMP, DB_USER, ACTION_NAME, RETURNCODE FROM ORACLE_AUDIT_TRAIL WHERE RETURNCODE!=0 ORDER BY TIMEST...
  ✅ Score     : 100.0%  |  SQL valide : Oui

[Test 4/8] [Moyen]
  ❓ Question : Quels utilisateurs Oracle ont généré le plus d'événements d'audit ?
  🔷 SQL généré  : SELECT DB_USER, COU

In [9]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 10 — TEST INTERACTIF : SQL / Résultat → Français    ║
# ║  Testez la reformulation de résultats Oracle bruts           ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Cas de test avec résultats SQL*Plus simulés ───────────────
SQL_TO_FR_TESTS = [
    {
        "label": "Connexions avec erreur ORA-01017",
        "sql": "SELECT DISTINCT DB_USER, OS_HOST FROM ORACLE_AUDIT_TRAIL WHERE RETURNCODE=1017;",
        "sqlplus_result": "DB_USER    OS_HOST\n---------  ---------------\nSCOTT      app-server-01\nHR         srv-oracle-02",
        "expected_keywords": ["SCOTT","HR","erreur","mot de passe","ORA-01017","connexion"],
        "difficulty": "Moyen"
    },
    {
        "label": "Activité par utilisateur",
        "sql": "SELECT DB_USER, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY DB_USER ORDER BY NB DESC;",
        "sqlplus_result": "DB_USER    NB\n---------  --\nSYS        87\nSCOTT      45\nHR         23\nAPP_USER   12",
        "expected_keywords": ["SYS","87","actif","utilisateur","actions"],
        "difficulty": "Facile"
    },
    {
        "label": "Activité nocturne suspecte",
        "sql": "SELECT DB_USER, ACTION_NAME, TIMESTAMP FROM ORACLE_AUDIT_TRAIL WHERE TO_NUMBER(SUBSTR(TIMESTAMP,12,2)) >= 22;",
        "sqlplus_result": "DB_USER  ACTION_NAME  TIMESTAMP\nSCOTT    SELECT       2026-02-10 23:14:22\nSYS      LOGON        2026-02-12 22:05:11",
        "expected_keywords": ["22h","nocturne","SCOTT","SYS","suspect","connexion"],
        "difficulty": "Difficile"
    },
    {
        "label": "Résumé global audit",
        "sql": "SELECT COUNT(*) TOTAL, SUM(CASE WHEN RETURNCODE=0 THEN 1 ELSE 0 END) SUCCES, SUM(CASE WHEN RETURNCODE!=0 THEN 1 ELSE 0 END) ERREURS FROM ORACLE_AUDIT_TRAIL;",
        "sqlplus_result": "TOTAL  SUCCES  ERREURS\n-----  ------  -------\n500    432     68",
        "expected_keywords": ["500","432","68","erreur","succès","événements"],
        "difficulty": "Moyen"
    },
    {
        "label": "Objets sensibles consultés",
        "sql": "SELECT DISTINCT DB_USER FROM ORACLE_AUDIT_TRAIL WHERE OBJ_NAME IN ('DBA_USERS','V$SESSION');",
        "sqlplus_result": "DB_USER\n-------\nSYS\nDBA_OPS",
        "expected_keywords": ["SYS","DBA_OPS","DBA_USERS","V$SESSION","sensible","système"],
        "difficulty": "Difficile"
    },
]

def score_sqlplus_response(response: str, expected_keywords: list) -> dict:
    resp_lower = response.lower()
    matched = [kw for kw in expected_keywords if kw.lower() in resp_lower]
    score = len(matched) / len(expected_keywords) * 100
    return {"score": round(score,1), "matched": matched,
            "missing": [kw for kw in expected_keywords if kw.lower() not in resp_lower]}

# ── Tests automatiques SQL*Plus → FR ─────────────────────────
print("="*65)
print("  TEST AUTOMATIQUE — SQL / Résultat Oracle → Français")
print("="*65)

results_sqlfr = []
total_sqlfr = 0

for i, test in enumerate(SQL_TO_FR_TESTS, 1):
    prompt = (
        f"Traduis ce résultat d'exécution SQL Oracle en français clair pour un non-technicien.\n"
        f"Requête SQL : {test['sql']}\n"
        f"Résultat Oracle SQL*Plus :\n{test['sqlplus_result']}"
    )
    print(f"\n[Test {i}/{len(SQL_TO_FR_TESTS)}] [{test['difficulty']}] {test['label']}")
    print(f"  📋 SQL       : {test['sql'][:80]}...")
    print(f"  📊 Résultat  :\n{test['sqlplus_result']}")

    response = generate(prompt, max_new_tokens=120, temperature=0.3)
    scoring  = score_sqlplus_response(response, test["expected_keywords"])

    emoji = "✅" if scoring["score"] >= 60 else ("⚠️" if scoring["score"] >= 30 else "❌")
    print(f"  💬 Réponse   : {response[:150]}{'...' if len(response)>150 else ''}")
    print(f"  {emoji} Score    : {scoring['score']}%")
    if scoring["missing"]:
        print(f"  ⚠️  Manquants : {scoring['missing']}")

    results_sqlfr.append({"label": test["label"], "score": scoring["score"],
                           "difficulty": test["difficulty"]})
    total_sqlfr += scoring["score"]

avg_sqlfr = total_sqlfr / len(SQL_TO_FR_TESTS)
print("\n" + "="*65)
print("  BILAN DES TESTS SQL→FR")
print("="*65)
for r in results_sqlfr:
    bar = "█" * int(r["score"]/10)
    emoji = "✅" if r["score"]>=60 else ("⚠️" if r["score"]>=30 else "❌")
    print(f"  {emoji} {r['label']:<35} {r['score']:>5.1f}%  {bar}")
perf = "🏆 Excellent" if avg_sqlfr>=80 else ("✅ Bon" if avg_sqlfr>=60 else ("⚠️ Moyen" if avg_sqlfr>=40 else "❌ À améliorer"))
print(f"  {'─'*50}")
print(f"  Score global : {avg_sqlfr:.1f}%  {perf}")
print("="*65)

# ── Zone de test libre SQL*Plus → FR ─────────────────────────
print("\n📝 ZONE DE TEST LIBRE — Entrez votre propre résultat Oracle :")
print("   Modifiez les variables ci-dessous et réexécutez la cellule.\n")

ma_requete_sql = "SELECT ACTION_NAME, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY ACTION_NAME ORDER BY NB DESC;"
mon_resultat_sqlplus = """ACTION_NAME     NB
-----------     --
SELECT          210
LOGON           142
LOGOFF          138
UPDATE           45
DELETE           12"""

prompt_libre = (
    f"Traduis ce résultat d'exécution SQL Oracle en français clair pour un non-technicien.\n"
    f"Requête SQL : {ma_requete_sql}\n"
    f"Résultat Oracle SQL*Plus :\n{mon_resultat_sqlplus}"
)
print(f"  📋 Votre requête : {ma_requete_sql}")
print(f"  📊 Votre résultat :\n{mon_resultat_sqlplus}")
reponse_libre = generate(prompt_libre, max_new_tokens=150, temperature=0.3)
print(f"\n  💬 Reformulation : {reponse_libre}")

  TEST AUTOMATIQUE — SQL / Résultat Oracle → Français

[Test 1/5] [Moyen] Connexions avec erreur ORA-01017
  📋 SQL       : SELECT DISTINCT DB_USER, OS_HOST FROM ORACLE_AUDIT_TRAIL WHERE RETURNCODE=1017;...
  📊 Résultat  :
DB_USER    OS_HOST
---------  ---------------
SCOTT      app-server-01
HR         srv-oracle-02
  💬 Réponse   : L'audit Oracle identifie les utilisateurs et serveurs ayant généré des erreurs ORA-01017, c'est-à-dire des tentatives de connexion avec un mot de pass...
  ✅ Score    : 66.7%
  ⚠️  Manquants : ['SCOTT', 'HR']

[Test 2/5] [Facile] Activité par utilisateur
  📋 SQL       : SELECT DB_USER, COUNT(*) AS NB FROM ORACLE_AUDIT_TRAIL GROUP BY DB_USER ORDER BY...
  📊 Résultat  :
DB_USER    NB
---------  --
SYS        87
SCOTT      45
HR         23
APP_USER   12
  💬 Réponse   : Cette requête affiche le volume total d'activité par utilisateur Oracle dans l'audit, avec la société SYSTEM et l'usager SCOTT au plus actif.
  ✅ Score    : 60.0%
  ⚠️  Manquants : ['87', 'action

In [10]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELLULE 11 — BILAN GLOBAL DE PERFORMANCE DU MODÈLE          ║
# ╚══════════════════════════════════════════════════════════════╝
import pandas as pd
from datetime import datetime

# Récupération des scores des cellules précédentes
# (réexécuter cellules 9 et 10 avant celle-ci)
try:
    scores_fr_sql = {r["difficulty"]: r["score"] for r in results}
    avg_fr_sql    = sum(r["score"] for r in results) / len(results)
    avg_sql_fr    = sum(r["score"] for r in results_sqlfr) / len(results_sqlfr)
    score_global  = (avg_fr_sql + avg_sql_fr) / 2
except:
    avg_fr_sql = avg_sql_fr = score_global = 0.0
    print("⚠️  Exécutez d'abord les cellules 9 et 10.")

print("╔══════════════════════════════════════════════════════════╗")
print("║          RAPPORT DE PERFORMANCE — MODÈLE NLP→SQL         ║")
print(f"║          Généré le : {datetime.now().strftime('%Y-%m-%d %H:%M')}                    ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  CONFIGURATION                                            ║")
print(f"║    Modèle base     : TinyLlama-1.1B-Chat-v1.0            ║")
print(f"║    Méthode         : LoRA Fine-Tuning                    ║")
print(f"║    Table testée    : ORACLE_AUDIT_TRAIL (500 lignes)     ║")
print(f"║    Dataset         : 1500 FR→SQL + 1500 SQL→FR           ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  RÉSULTATS                                                ║")
print(f"║    FR → SQL Oracle : {avg_fr_sql:>5.1f}%                             ║")
print(f"║    SQL/Résultat→FR : {avg_sql_fr:>5.1f}%                             ║")
print(f"║    Score global    : {score_global:>5.1f}%                             ║")

# Interprétation
if score_global >= 80:
    verdict = "🏆 EXCELLENT — Prêt pour production"
    details = "Le modèle génère du SQL Oracle précis et des reformulations naturelles."
elif score_global >= 65:
    verdict = "✅ BON — Utilisable avec supervision"
    details = "Performances satisfaisantes. Quelques cas limites à améliorer."
elif score_global >= 45:
    verdict = "⚠️  MOYEN — Dataset plus large recommandé"
    details = "Le modèle comprend les requêtes simples mais échoue sur les cas complexes."
else:
    verdict = "❌ INSUFFISANT — Entraînement supplémentaire nécessaire"
    details = "Augmenter le dataset et les epochs d'entraînement."

print(f"║  VERDICT   : {verdict:<44}║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  RECOMMANDATIONS                                          ║")
print(f"║    {details[:55]:<55}║")
print("╠══════════════════════════════════════════════════════════╣")
print("║  PROCHAINES ÉTAPES VERS PRODUCTION                       ║")
steps = [
    "1. Quantization 4-bit (bitsandbytes) pour CPU",
    "2. Export GGUF (llama.cpp) pour déploiement léger",
    "3. API FastAPI + validation SQL (SELECT uniquement)",
    "4. Interface web FastAPI + Jinja2 ou React",
    "5. Tests sur vraie base Oracle (cx_Oracle / SQLAlchemy)",
    "6. Monitoring et logs d'audit des requêtes générées",
]
for step in steps:
    print(f"║    {step:<54}║")
print("╚══════════════════════════════════════════════════════════╝")

# Export CSV du rapport
report_data = []
try:
    for r in results:
        report_data.append({"sens":"FR→SQL","test":r["question"][:60],
                             "score":r["score"],"difficulte":r["difficulty"]})
    for r in results_sqlfr:
        report_data.append({"sens":"SQL→FR","test":r["label"],
                             "score":r["score"],"difficulte":r["difficulty"]})
    pd.DataFrame(report_data).to_csv("rapport_performance_modele.csv", index=False)
    print("\n📄 Rapport exporté : rapport_performance_modele.csv")
except:
    pass

╔══════════════════════════════════════════════════════════╗
║          RAPPORT DE PERFORMANCE — MODÈLE NLP→SQL         ║
║          Généré le : 2026-02-26 11:24                    ║
╠══════════════════════════════════════════════════════════╣
║  CONFIGURATION                                            ║
║    Modèle base     : TinyLlama-1.1B-Chat-v1.0            ║
║    Méthode         : LoRA Fine-Tuning                    ║
║    Table testée    : ORACLE_AUDIT_TRAIL (500 lignes)     ║
║    Dataset         : 1500 FR→SQL + 1500 SQL→FR           ║
╠══════════════════════════════════════════════════════════╣
║  RÉSULTATS                                                ║
║    FR → SQL Oracle :  87.5%                             ║
║    SQL/Résultat→FR :  62.0%                             ║
║    Score global    :  74.8%                             ║
║  VERDICT   : ✅ BON — Utilisable avec supervision         ║
╠══════════════════════════════════════════════════════════╣
║  RECOMMANDATIONS       

## 🚀 Déploiement Local & Production

### Commandes pip (environnement local)

```bash
python -m venv venv_oracle
source venv_oracle/bin/activate  # Linux/Mac

pip install --upgrade numpy
pip install transformers>=4.40.0 peft>=0.10.0 accelerate>=0.28.0 datasets>=2.19.0
pip install torch>=2.2.0 --index-url https://download.pytorch.org/whl/cu121
pip install huggingface_hub>=0.22.0 pandas>=2.0.0 bitsandbytes fastapi uvicorn
```

### Fichiers à récupérer depuis Colab

| Fichier | Description |
|---|---|
| `TinyLlama-1.1B-Chat-v1.0.zip` | Modèle de base |
| `tinyllama_oracle_lora.zip` | Adapter LoRA fine-tuné |
| `oracle_audit_trail.csv` | Table d'audit Oracle simulée |
| `oracle_nlp_dataset.csv` | Dataset d'entraînement (3000 ex.) |
| `rapport_performance_modele.csv` | Résultats de performance |

### Architecture de production

```
Utilisateur (question FR)
       ↓
Interface Web (FastAPI + Jinja2)
       ↓
API /query  →  Modèle TinyLlama + LoRA (4-bit CPU)
       ↓
Validateur SQL (SELECT uniquement, anti-injection)
       ↓
Oracle DB (cx_Oracle, compte lecture seule)
       ↓
Modèle TinyLlama + LoRA  →  Reformulation FR
       ↓
Réponse affichée à l'utilisateur
```

### Quantization 4-bit pour CPU (production)

```python
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
quantization_config = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-Chat-v1.0",
    quantization_config=quantization_config,
    device_map="cpu"
)
```